In [ ]:
# Check if the directly connected customers have all its prefixes on their paths.

import pybgpstream
import pandas as pd
from multiprocessing import Pool, cpu_count

scrubber = "32787"  # Replace with the actual ASN you want to exclude
year = "2024"


def has_other_upstream(asn):
    """
    Check if a given prefix has an upstream ASN other than the scrubber.
    """
    stream = pybgpstream.BGPStream(
        from_time=year+"-01-01 00:00:00 UTC",
        until_time=year+"-01-01 00:00:00 UTC",
        record_type="ribs",   
        filter="path !_"+scrubber+"_ and path "+str(asn)+"$"
      
    )

    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    for rec in stream.records():
        for elem in rec:
            if elem:
                origin = elem.fields["as-path"].split()[-1]
                prefix = elem.fields["prefix"]
                if prefix != "0.0.0.0/0":
#                     print(elem)
                    return origin  # Return origin asn if another upstream exists
    return None  # Return None if no other upstream found



df = pd.read_csv("../data/confirmed_customers_as"+scrubber+"_"+year+".csv")
asns = df["origin_as"].unique()

"""
Main function to run prefix checks in parallel.
"""
#     with Pool(processes=cpu_count()) as pool:
with Pool(processes=6) as pool:
    results = pool.map(has_other_upstream, asns)

 # Filter out None values (prefixes that had no other upstream)
valid_asns = [origin_as for origin_as in results if origin_as]
valid_asns_unique = list(set(valid_asns))

# Count how many prefixes have other ASNs as upstream
count = len(valid_asns_unique)

print(f"Total asns with other upstreams: {count}   out of {len(asns)} asns.")
print("List of asns that have other upstreams for its prefixes:", valid_asns_unique)


In [14]:
# Check if the directly connected customer prefixes have other ASN except scrubber as an upstream on their paths.

import pybgpstream
import pandas as pd
from multiprocessing import Pool, cpu_count

scrubber = "19905"  # Replace with the actual ASN you want to exclude
year = "2024"


def has_other_upstream(prefix):
    """
    Check if a given prefix has an upstream ASN other than the scrubber.
    """
    stream = pybgpstream.BGPStream(
        from_time=year+"-01-01 00:00:00 UTC",
        until_time=year+"-01-01 00:00:00 UTC",
        record_type="ribs",   
        filter="path !_"+scrubber+"_ and prefix exact "+prefix
      
    )

    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    for rec in stream.records():
        for elem in rec:
            if elem:
                prefix = elem.fields["prefix"]
                provider = elem.fields["as-path"].split()[-2]
                origin = elem.fields["as-path"].split()[-1]

                 # Check if the origin AS is path prepending
                if provider != origin and provider != scrubber:
                    print(f"No path prepending and a different upstream {elem}")
                    return prefix  # Return origin asn if another upstream exists
    return None  # Return None if no other upstream found



df = pd.read_csv("../data/confirmed_customers_as"+scrubber+"_"+year+".csv")
prefixes = df["prefix"].unique()

"""
Main function to run prefix checks in parallel.
"""
#     with Pool(processes=cpu_count()) as pool:
with Pool(processes=6) as pool:
    results = pool.map(has_other_upstream, prefixes)

 # Filter out None values (prefixes that had no other upstream)
valid_prefixes = [prefix for prefix in results if prefix]
                      
                              
valid_prefixes_unique = list(set(valid_prefixes))

# Count how many prefixes have other ASNs as upstream
count = len(valid_prefixes)

print(f"Total prefixes with other upstreams: {count}   out of {len(prefixes)} prefixes.")
print("List of asns that have other upstreams for its prefixes:", valid_prefixes_unique)

No path prepending and a different upstream rib|R|1704067200.000000|ris|rrc13|None|None|24482|195.208.209.109|94.136.40.0/24|195.208.209.109|24482 20773 20773 20773 20773 20773 20738|24482:200 24482:12000 24482:2 24482:12022 24482:12020 6695:6695 20738:44 24482:65203|None|None
No path prepending and a different upstream rib|R|1704067200.000000|ris|rrc15|None|None|47787|187.16.213.225|185.130.132.0/24|187.16.213.225|47787 8400 43281 203571|47787:20000 47787:21130 47787:3120 47787:1010|None|None
No path prepending and a different upstream rib|R|1704067200.000000|ris|rrc15|None|None|47787|187.16.213.225|185.130.133.0/24|187.16.213.225|47787 8400 43281 203571|47787:20000 47787:21130 47787:3120 47787:1010|None|None


KeyboardInterrupt: 

In [8]:
len(prefixes)

5084